In [3]:
import wandb
import polars as pl
from pathlib import Path

In [4]:
api = wandb.Api()

ARTIFACTS = {
    "raw-site-metadata": {"table": "site_metadata", "partitioned": False, "nested": True},
    "raw-watershed-mapping": {
        "table": "nldas3_watershed_mapping",
        "partitioned": False,
        "nested": True,
    },
    "raw-streamflow-daily": {"table": "streamflow_daily", "partitioned": False, "nested": True},
    "raw-nldas3-forcing": {"table": "nldas3_forcing", "partitioned": True, "nested": True},
    "raw-streamflow-15min": {"table": "streamflow_15min", "partitioned": True, "nested": True},
    "flood-dataset": {"table": "flood_model", "partitioned": False, "nested": False},
    "flood-dataset-daily": {"table": "flood_model_daily", "partitioned": False, "nested": False},
}

dfs = {}
for artifact_name, cfg in ARTIFACTS.items():
    print(f"Downloading {artifact_name}...")
    artifact = api.artifact(f"flood-forecasting/{artifact_name}:latest")
    artifact_dir = Path(artifact.download())

    table = cfg["table"]
    if cfg["partitioned"]:
        parquet_files = sorted(artifact_dir.glob(f"{table}/{table}_*.parquet"))
        df = pl.concat([pl.read_parquet(f) for f in parquet_files])
    elif cfg["nested"]:
        df = pl.read_parquet(artifact_dir / table / f"{table}.parquet")
    else:
        df = pl.read_parquet(artifact_dir / f"{table}.parquet")

    dfs[table] = df
    print(f"  {table}: {len(df):,} rows")

wandb:   1 of 1 files downloaded.  


  site_metadata: 3,430 rows


wandb:   1 of 1 files downloaded.  


  nldas3_watershed_mapping: 3,430 rows


wandb:   1 of 1 files downloaded.  


...

  streamflow_15min: 498,403,182 rows


wandb: Downloading large artifact 'flood-dataset:latest', 4522.32MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:58.2 (77.7MB/s)


  flood_model: 101,651,130 rows


wandb:   1 of 1 files downloaded.  


  flood_model_daily: 5,019,758 rows


In [5]:
print(f"{'Table':<30} {'Rows':>15}")
print("-" * 47)

for table, df in dfs.items():
    print(f"{table:<30} {len(df):>15,}")

Table                                     Rows
-----------------------------------------------
site_metadata                            3,430
nldas3_watershed_mapping                 3,430
streamflow_daily                     5,015,435
nldas3_forcing                     549,513,440
streamflow_15min                   498,403,182
flood_model                        101,651,130
flood_model_daily                    5,019,758
